In [2]:
import numpy as np
from scipy import stats
import pandas as pd

In [5]:
params = {
    'Container': {'lam': 5.5812, 'mu': 9.6069, 'sigma': 0.3493},
    'Tanker': {'lam': 2.2299, 'mu': 10.9119, 'sigma': 0.2963},
    'Bulk': {'lam': 2.2947, 'mu': 10.4989, 'sigma': 0.3806},
}

days_in_year = 365
nr_runs = 10000
rng = np.random.default_rng(42)
def simPoissonProcess(lam, t, rng):
    intArrivalTimsDist = stats.expon(scale=1/lam)
    Nt = 0
    time = intArrivalTimsDist.rvs(random_state=rng)
    while time < t:
        Nt += 1
        time += intArrivalTimsDist.rvs(random_state=rng)
    return Nt


results_for_all_runs = {k: np.zeros(nr_runs) for k in 
                        ['NC', 'NT', 'NB', 'N', 'SC', 'ST', 'SB', 'S']}

for r in range(nr_runs): 
    NC = simPoissonProcess(params['Container']['lam'], days_in_year, rng)
    NT = simPoissonProcess(params['Tanker']['lam'], days_in_year, rng)
    NB = simPoissonProcess(params['Bulk']['lam'], days_in_year, rng)

    N = NC + NT + NB

    SC = stats.lognorm(s=params['Container']['sigma'], scale=np.exp(params['Container']['mu'])).rvs(size=NC, random_state=rng).sum()
    ST = stats.lognorm(s=params['Tanker']['sigma'], scale=np.exp(params['Tanker']['mu'])).rvs(size=NT, random_state=rng).sum()
    SB = stats.lognorm(s=params['Bulk']['sigma'], scale=np.exp(params['Bulk']['mu'])).rvs(size=NB, random_state=rng).sum()

    S = SC + ST + SB

    results_for_all_runs['NC'][r],results_for_all_runs['NT'][r], results_for_all_runs['NB'][r] = NC, NT, NB
    results_for_all_runs['N'][r] = N
    results_for_all_runs['SC'][r], results_for_all_runs['ST'][r], results_for_all_runs['SB'][r] = SC, ST, SB
    results_for_all_runs['S'][r] = S

# Table 2
rows = []
for key in ['NC', 'NT', 'NB', 'N', 'SC', 'ST', 'SB', 'S']:
    x = results_for_all_runs[key]
    rows.append([key, x.mean(), x.std(ddof=1), np.percentile(x, 95)])

table2 = pd.DataFrame(rows, columns=['Quantity', 'Mean', 'Standard deviation', '95% quantile'])


    



In [6]:
table2


,Quantity,Mean,Standard deviation,95% quantile
0,NC,2.037658e+03,4.563067e+01,2.114000e+03
1,NT,8.132826e+02,2.885329e+01,8.610000e+02
2,NB,8.378428e+02,2.888171e+01,8.860000e+02
3,N,3.688784e+03,6.074481e+01,3.789000e+03
4,SC,3.219782e+07,7.655626e+05,3.346905e+07
5,ST,4.659350e+07,1.727383e+06,4.943325e+07
6,SB,3.268192e+07,1.206336e+06,3.466436e+07
7,S,1.114732e+08,2.224294e+06,1.151669e+08


In [ ]:

T = 365
types = ['Container', 'Tanker', 'Bulk']

# Theoretical values from compound Poisson formulas
theo = {}

# Arrival counts: E[N_i] = λ_i T,  Var[N_i] = λ_i T
for t in types:
    lam = params[t]['lam']
    theo[f'N_{t[0]}(T)_mean'] = lam * T
    theo[f'N_{t[0]}(T)_std']  = np.sqrt(lam * T)

# Total arrivals: sum of independent Poissons
lam_total = sum(params[t]['lam'] for t in types)
theo['N(T)_mean'] = lam_total * T
theo['N(T)_std']  = np.sqrt(lam_total * T)

# Cargo per type: E[S_i] = λ_i T · E[X], Var[S_i] = λ_i T · E[X²]
def EX(mu, sigma):
    return np.exp(mu + sigma**2 / 2)

def EX2(mu, sigma):
    return np.exp(2 * mu + 2 * sigma**2)

for t in types:
    lam = params[t]['lam']
    mu  = params[t]['mu']
    sg  = params[t]['sigma']
    theo[f'S_{t[0]}(T)_mean'] = lam * T * EX(mu, sg)
    theo[f'S_{t[0]}(T)_std']  = np.sqrt(lam * T * EX2(mu, sg))

# Total cargo: sum of independent compound Poissons
theo['S(T)_mean'] = sum(theo[f'S_{t[0]}(T)_mean'] for t in types)
theo['S(T)_std']  = np.sqrt(sum(theo[f'S_{t[0]}(T)_std']**2 for t in types))

# Simulated values from results dict 
sim = {k: results_for_all_runs[k].mean() for k in results_for_all_runs}
sim_std = {k: results_for_all_runs[k].std(ddof=1) for k in results_for_all_runs}

# Build the comparison table (with percentage)
rows = []
mapping = [
    ('E[N_C(T)]',   'NC', 'N_C(T)_mean'),
    ('Std[N_C(T)]', 'NC', 'N_C(T)_std'),
    ('E[N_T(T)]',   'NT', 'N_T(T)_mean'),
    ('Std[N_T(T)]', 'NT', 'N_T(T)_std'),
    ('E[N_B(T)]',   'NB', 'N_B(T)_mean'),
    ('Std[N_B(T)]', 'NB', 'N_B(T)_std'),
    ('E[N(T)]',     'N',  'N(T)_mean'),
    ('Std[N(T)]',   'N',  'N(T)_std'),
    ('E[S_C(T)]',   'SC', 'S_C(T)_mean'),
    ('Std[S_C(T)]', 'SC', 'S_C(T)_std'),
    ('E[S_T(T)]',   'ST', 'S_T(T)_mean'),
    ('Std[S_T(T)]', 'ST', 'S_T(T)_std'),
    ('E[S_B(T)]',   'SB', 'S_B(T)_mean'),
    ('Std[S_B(T)]', 'SB', 'S_B(T)_std'),
    ('E[S(T)]',     'S',  'S(T)_mean'),
    ('Std[S(T)]',   'S',  'S(T)_std'),
]

for label, sim_key, theo_key in mapping:
    theoretical = theo[theo_key]
    simulated   = sim[sim_key] if 'mean' in theo_key else sim_std[sim_key]
    diff_pct    = (simulated - theoretical) / theoretical * 100
    rows.append([label, theoretical, simulated, diff_pct])

validation = pd.DataFrame(rows,
    columns=['Quantity', 'Theoretical', 'Simulated', 'Difference (%)'])

print(validation.to_string(index=False, float_format='{:,.2f}'.format))

   Quantity    Theoretical      Simulated  Difference (%)
  E[N_C(T)]       2,037.14       2,037.66            0.03
Std[N_C(T)]          45.13          45.63            1.10
  E[N_T(T)]         813.91         813.28           -0.08
Std[N_T(T)]          28.53          28.85            1.14
  E[N_B(T)]         837.57         837.84            0.03
Std[N_B(T)]          28.94          28.88           -0.20
    E[N(T)]       3,688.62       3,688.78            0.00
  Std[N(T)]          60.73          60.74            0.02
  E[S_C(T)]  32,191,287.70  32,197,819.74            0.02
Std[S_C(T)]     758,092.79     765,562.56            0.99
  E[S_T(T)]  46,625,163.38  46,593,497.49           -0.07
Std[S_T(T)]   1,707,636.38   1,727,383.40            1.16
  E[S_B(T)]  32,665,422.14  32,681,918.79            0.05
Std[S_B(T)]   1,213,483.97   1,206,336.38           -0.59
    E[S(T)] 111,481,873.22 111,473,236.02           -0.01
  Std[S(T)]   2,227,839.77   2,224,293.73           -0.16


In [10]:
# Table of arrival share VS tonnage share 

arrivals = {
    'Container': results_for_all_runs['NC'].mean(),
    'Tanker': results_for_all_runs['NT'].mean(),
    'Bulk': results_for_all_runs['NB'].mean(),
}

tonnage = {
    'Container': results_for_all_runs['SC'].mean(),
    'Tanker': results_for_all_runs['ST'].mean(),
    'Bulk': results_for_all_runs['SB'].mean(),
}

total_arrivals = sum(arrivals.values())
total_tonnage = sum(tonnage.values())

share_compare = pd.DataFrame({
    'Vessel type': list(arrivals.keys()),
    'Arrivals': [arrivals[t] for t in arrivals],
    'Tonnage(Million tons)': [tonnage[t] / 1e6 for t in tonnage],
})

share_compare['Arrival share(%)'] = (share_compare['Arrivals'] / total_arrivals * 100).round(2)
share_compare['Tonnage share(%)'] = (share_compare['Tonnage(Million tons)'] / (total_tonnage/1e6) * 100).round(2)

print(share_compare.to_string(index=False, float_format='{:,.2f}'.format))

Vessel type  Arrivals  Tonnage(Million tons)  Arrival share(%)  Tonnage share(%)
  Container  2,037.66                  32.20             55.24             28.88
     Tanker    813.28                  46.59             22.05             41.80
       Bulk    837.84                  32.68             22.71             29.32
